# Construye agentes de IA con Pydantic AI
**FISC · 24 de septiembre de 2026 · Ricardo Tovar**

En 40 minutos: ejecuta un agente, agrega una herramienta, valida su salida y continúa una conversación. Necesitas Python básico, Google Colab y una clave propia de Google AI Studio. El uso de la API puede estar sujeto a cuotas o cobros de tu cuenta.

**No pegues ni compartas tu clave en una celda.** Guarda un secreto llamado `GOOGLE_API_KEY` en el panel de Secrets de Colab y habilita el acceso del notebook.

## 0 · Prepara el entorno (5 min)
Ejecuta la instalación. Si Colab pide reiniciar el entorno, hazlo antes de seguir.

In [ ]:
%pip install -q pydantic-ai

In [ ]:
import os
from google.colab import userdata

clave = userdata.get('GOOGLE_API_KEY')
if not clave:
    raise RuntimeError('Crea GOOGLE_API_KEY en Colab Secrets y habilita el acceso del notebook.')
os.environ['GOOGLE_API_KEY'] = clave
del clave
print('Clave disponible para esta sesión; no se mostrará su valor.')

## 1 · Primer agente (7 min)
El modelo se puede cambiar si tu cuenta no tiene acceso al sugerido. Ejecuta la celda, cambia una palabra de la instrucción y observa el resultado.

In [ ]:
from pydantic_ai import Agent

MODELO = 'google:gemini-3.7-flash'
agente_simple = Agent(MODELO, instructions='Responde en español, en una sola oración clara.')
primera = agente_simple.run_sync('¿Qué es una herramienta de un agente de IA?')
print(primera.output)

## 2 · Dale una herramienta (10 min)
Una tool consulta datos definidos por nuestro programa. El agente puede decidir cuándo llamarla; en esta práctica le pedimos que la use antes de responder. La guía de abajo es ficticia y local: no hay búsqueda en internet.

In [ ]:
GUIA = {
    'horario': 'El taller ocurre el jueves 24 de septiembre de 2026, de 1:00 a 3:00 p. m.',
    'requisito': 'Se requiere Python básico, Google Colab y una clave propia de Google AI Studio.',
    'entrega': 'La práctica termina con una ejecución reproducible, un caso ambiguo, un fallo observado y una mejora concreta.',
}

agente_guia = Agent(
    MODELO,
    instructions=(
        'Responde sólo con la guía FISC. Antes de responder, usa consultar_guia. '
        'Si no hay dato, dilo sin inventar. Indica el tema consultado.'
    ),
)

@agente_guia.tool_plain
def consultar_guia(tema: str) -> str:
    """Busca en la guía local. Temas: horario, requisito o entrega."""
    tema = tema.strip().lower()
    print(f'Tool consultada: {tema}')
    return GUIA.get(tema, 'No hay información sobre ese tema en la guía.')

consulta = agente_guia.run_sync('¿A qué hora es el taller?')
print(consulta.output)

**Prueba:** pregunta por el lugar exacto del taller. ¿La respuesta admite que falta el dato? Si el agente escoge otro tema, ajusta la descripción de la tool o la instrucción y vuelve a probar.

In [ ]:
sin_dato = agente_guia.run_sync('¿En qué salón exacto será el taller?')
print(sin_dato.output)

## 3 · Define una salida tipada (8 min)
`output_type` indica el contrato que Pydantic valida. La validación comprueba estructura y tipos; no demuestra por sí sola que los hechos sean correctos. Contrasta la salida con la guía.

In [ ]:
from pydantic import BaseModel, Field

class RespuestaGuia(BaseModel):
    respuesta: str = Field(description='Respuesta breve basada en la guía')
    tema_consultado: str = Field(description='Tema buscado con la herramienta')
    dato_disponible: bool = Field(description='Si la guía contiene el dato solicitado')

agente_tipado = Agent(
    MODELO,
    output_type=RespuestaGuia,
    instructions=(
        'Responde sólo con la guía FISC. Usa consultar_guia antes de responder. '
        'Si la guía no contiene el dato solicitado, marca dato_disponible como false '
        'y explica que no se conoce. Nunca inventes un salón o una persona.'
    ),
)

@agente_tipado.tool_plain
def consultar_guia(tema: str) -> str:
    """Busca en la guía local. Temas: horario, requisito o entrega."""
    return GUIA.get(tema.strip().lower(), 'No hay información sobre ese tema en la guía.')

respuesta = agente_tipado.run_sync('¿En qué salón exacto será el taller?')
print(respuesta.output.model_dump())
print(type(respuesta.output).__name__)

## 4 · Continúa con historial (5 min)
`all_messages()` contiene los mensajes de la primera ejecución. Pasarlos como `message_history` da contexto al siguiente turno. Este historial vive sólo en esta sesión de Colab; no es memoria persistente.

In [ ]:
turno_1 = agente_tipado.run_sync('¿Qué necesito para participar?')
print('Turno 1:', turno_1.output.model_dump())
turno_2 = agente_tipado.run_sync(
    '¿Y cuál es la entrega?',
    message_history=turno_1.all_messages(),
)
print('Turno 2:', turno_2.output.model_dump())

## 5 · Tu cambio (5 min)
1. Añade a `GUIA` el tema `material` con un dato breve y verdadero del taller.
2. Pregunta por ese material y observa la salida tipada.
3. Haz una pregunta ambigua o fuera de la guía. Anota un fallo observado y cambia una sola instrucción o descripción de tool.
4. Vuelve a ejecutar y compara. El resultado del modelo puede variar entre ejecuciones.

**Registro rápido:** caso probado: ______ · resultado esperado: ______ · observado: ______ · cambio: ______ · nuevo resultado: ______

## Después del taller
Elige un proyecto pequeño: extractor de acuerdos, asistente sobre una guía propia o clasificador de incidencias. Conserva una ejecución reproducible, un caso ambiguo, un fallo observado y una mejora.

Documentación: [Pydantic AI](https://pydantic.dev/docs/ai/overview/) · [Google](https://pydantic.dev/docs/ai/models/google/) · [Historial](https://pydantic.dev/docs/ai/core-concepts/message-history/) · [Tools](https://pydantic.dev/docs/ai/tools-toolsets/tools/).

© Ricardo Tovar, 2026. Descarga y modifica este notebook para estudio personal. La republicación y su uso en otra charla requieren autorización expresa.